# MIDAS Civil NX API

**Who this is for.** Engineers driving MIDAS Civil NX from Python — reading a
model, building one programmatically, running analysis, and pulling results —
through the official MAPI REST interface (never MCT files).

**What you need.** Civil NX running with the API enabled, and `~/secrets.json`
carrying `MIDAS_API_KEY` (plus `MIDAS_BASE_URL` if not the default local
endpoint). The model-*building* helpers below run fine without a live program —
they just produce the JSON payloads.

**Prev:** [14. BrDR and BrM Clients](14.%20BrDR%20and%20BrM%20Clients.ipynb)

## Step 1 — Connect

`MidasCivil()` is the handle to the running program; `ping()` confirms the link
before you do anything else. (On a machine without Civil NX this cell reports
the failure and the rest of the notebook still runs.)

In [1]:
import os, sys
import pandas as pd

sys.path.insert(0, os.path.abspath("."))   # run from the civilpy repo root

from src.civilpy.structural.midas import MidasCivil

try:
    midas = MidasCivil()
    if midas.ping():
        print("connected")
    else:
        midas = None
        print("Civil NX not reachable here — payload cells below still run.")
except Exception as e:
    midas = None
    print(f"Civil NX not reachable here ({type(e).__name__}) — payload cells below still run.")

Civil NX not reachable here — payload cells below still run.


## Step 2 — One pattern for the whole model

Everything in a MIDAS model is a named database table behind `get_db` /
`put_db` / `post_db` / `delete_db`, and the client wraps the common ones:

| read | write | what |
|---|---|---|
| `nodes()` | `put_nodes()` | joints |
| `elements()` | `put_elements()` | members |
| `materials()` | `put_materials()` | materials |
| `sections()` | `put_sections()` | cross sections |
| `supports()` | `put_supports()` | boundary conditions |
| `static_loads()` | `put_static_loads()` | load cases |
| `groups()`, `load_combinations()`, `units()` | `set_units()` | bookkeeping |

Plus lifecycle: `new()`, `open(path)`, `save()`, `save_as()`, `analyze()`,
`summarize()` (one-screen model overview), and `can_analyze`/`capability` to
check the license tier before you script something it can't do.

In [2]:
if midas:
    print(midas.summarize())
else:
    print("(connect to a model and midas.summarize() prints counts of nodes,\n"
          " elements, materials, sections, supports, and load cases)")

(connect to a model and midas.summarize() prints counts of nodes,
 elements, materials, sections, supports, and load cases)


## Step 3 — Build model pieces without touching the GUI

`civilpy.structural.midas_models` generates ready-to-`put_db` payloads.
Materials and sections first — note the AISC database from notebook 13 flows
straight in:

In [3]:
from src.civilpy.structural.midas_models import (
    concrete_material_block, rolled_i_section_block, steel_material_block,
    unit_block,
)

units = unit_block(force="KIPS", dist="FT")
steel = steel_material_block("A709-50")
conc = concrete_material_block("Class-S-4500", matl_id=2, fc_psi=4500)
sect = rolled_i_section_block("W36X150")       # AISC lookup -> MIDAS section

print("material:", steel["1"]["NAME"], "| type:", steel["1"]["TYPE"])
print("section :", sect["1"]["SECT_NAME"], "| SECTTYPE:", sect["1"]["SECTTYPE"])

material: A709-50 | type: USER
section : W36X150 | SECTTYPE: DBUSER


## Step 4 — Whole framing systems in one call

The geometry generators lay out nodes and elements for you. A horizontally
curved two-girder unit — chorded girder lines, optional diaphragms at every
station:

In [4]:
from src.civilpy.structural.midas_models import curved_girder_model

model = curved_girder_model(
    radius=600.0, central_angle_deg=30.0, n_segments=10,
    girder_offsets=[-6.0, 6.0],                # two girders, 12 ft apart
    diaphragm_sect=2,
)
print(f"{len(model['NODE'])} nodes, {len(model['ELEM'])} elements")
print("push with: midas.put_nodes(model['NODE']); midas.put_elements(model['ELEM'])")

22 nodes, 31 elements
push with: midas.put_nodes(model['NODE']); midas.put_elements(model['ELEM'])


`bifurcated_girder_model` does the same for a girder line that splits into
branches (ramp gores), `abutment_connection` wires girder ends to a seat with
bearing springs, and `soil_spring_supports` turns t-z/q-z style spring tables
into MIDAS boundary payloads.

## Step 5 — Load the Ohio rating vehicles

`midas_standard_vehicle` emits MIDAS's built-in AASHTO vehicles;
`load_oh_vehicles(client)` pushes the whole Ohio rating fleet (legal trucks,
SU series, HL-93/HS20) as moving-load vehicles in one call, sharing the axle
definitions from `civilpy.structural.aashto.vehicles`.

In [5]:
from src.civilpy.structural.midas_models import midas_standard_vehicle

hl93 = midas_standard_vehicle("HL-93")
{k: hl93[k] for k in ["VEHICLE_LOAD_NAME", "STANDARD_CODE", "VEHICLE_TYPE_NAME"]}

{'VEHICLE_LOAD_NAME': 'HL-93',
 'STANDARD_CODE': 'AASHTO-LRFD',
 'VEHICLE_TYPE_NAME': 'HL-93'}

## Step 6 — Push a whole `StructuralModel`

If you already have a civilpy `StructuralModel` (framing plan, line-girder
tool, the box-beam/PSI pipelines…), `push_midas(model)` converts and uploads
the entire thing — `midas_payloads(model)` gives you the payload dict if you
want to inspect before sending.

## Step 7 — Read results back

After `analyze()`, result tables come back as `{"HEAD": [...], "DATA": [...]}`
blocks. Three helpers turn them into numbers: `parse_result_table` (rows as
dicts), `column_values` (one column, cast), `envelope` (max/|max|).

In [6]:
from src.civilpy.structural.midas import column_values, envelope, parse_result_table

response = {"BeamForce": {                      # shape of a real /post/TABLE reply
    "HEAD": ["Elem", "Load", "Moment-y"],
    "DATA": [["1", "DC", "812.4"], ["2", "DC", "-955.1"], ["3", "DC", "640.2"]],
}}
rows = parse_result_table(response)
print(rows[0])
print("My values:", column_values(rows, "Moment-y"))
print("envelope :", envelope(rows, "Moment-y"))

{'Elem': '1', 'Load': 'DC', 'Moment-y': '812.4'}
My values: [812.4, -955.1, 640.2]
envelope : 955.1


**Where to go next:** [11. MIDAS Bridge Model Walkthrough](11.%20MIDAS%20Bridge%20Model%20Walkthrough.ipynb)
tours a real model end-to-end, and the four *Midas Design Guide* notebooks work
complete designs (PSC girder, steel composite, and both load-rating chapters).